In [112]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import set_config

set_config(transform_output="pandas")  # sklearn transformers return DataFrames with column names

In [113]:
data = pd.read_csv("../data/train_data.csv")

In [114]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(data, test_size=0.2, random_state=42)
print("labelled file:", data.shape, "| train:", train.shape, "| internal test (set aside):", test.shape)

labelled file: (143579, 14) | train: (114863, 14) | internal test (set aside): (28716, 14)


## Preprocessing (train only)

All preprocessing decisions and fits use `train` only; `test` is only transformed with what was learned on `train`.

### 1. Outliers (IQR)

Univariate detection with the IQR rule: values below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR. `Balance` is excluded: the IQR is not meaningful (54% zeros, two populations), and it is replaced by `has_balance` (binary, no outliers).

In [115]:
for col in ["Age", "CreditScore", "EstimatedSalary"]:
    q1, q3 = train[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_low = (train[col] < low).sum()
    n_high = (train[col] > high).sum()
    share = (n_low + n_high) / len(train)
    print(f"{col}: bounds [{low:.1f}, {high:.1f}] | below: {n_low} | above: {n_high} | share: {share:.2%}")

Age: bounds [17.0, 57.0] | below: 0 | above: 4473 | share: 3.89%
CreditScore: bounds [427.5, 879.5] | below: 191 | above: 0 | share: 0.17%
EstimatedSalary: bounds [-45900.2, 275338.7] | below: 0 | above: 0 | share: 0.00%


**Decision: no row is removed.** No flagged value is an error:
- `Age`: 4,473 customers above 57 (3.9%). Plausible ages for bank customers, and informative: churn drops after 60 (`01_eda` §18.1 c). Removing them would hide this pattern.
- `CreditScore`: 191 scores below 427.5 (0.2%). The scale is not documented, but values (350 to 850) are consistent with common credit score scales (e.g. FICO: 300 to 850): low scores are rare but possible, not errors.
- `EstimatedSalary`: no value flagged (almost uniform distribution between 0 and 200,000). The very low values (min 11.58) are implausible as annual salaries but not provably wrong; the variable is likely dropped at feature selection anyway (negligible link with churn).

### 2. Encoding

| Variable | Values | Ordered? | Encoding | Reason |
|---|---|---|---|---|
| `Geography` | France, Spain, Germany | no | One-Hot (`drop="first"`) | nominal; dropping one column avoids redundant columns for the logistic regression |
| `Gender` | Male, Female | no | `OneHotEncoder(drop="if_binary")`: one 0/1 column (`Gender_Male`) | two categories; keeping it is an ethical decision (feature selection) |
| `NumOfProducts` | 1, 2, 3, 4 | yes | group 3 and 4 into **3+**, then One-Hot (`drop="first"`) | 3 and 4 are rare (2% of customers) with the same churn (≈ 87%); the link is U-shaped (35% / 6% / 87%), so an ordinal code would impose a monotonic effect in a logistic regression, while One-Hot gives each category its own coefficient |

An ordinal code ordered by churn rate (2 < 1 < 3+) was also considered: it would fit the logistic regression reasonably well (almost equal steps in log-odds), but it uses the target to define the order and only saves one column. One-Hot is kept: exact, simpler, and it does not use the target. The same encoding works for decision trees.

### 3. Feature engineering

Two models are compared: a **logistic regression** and a **decision tree**.

| Feature | Logistic regression | Decision tree | Reason (EDA) |
|---|---|---|---|
| `has_balance` (1 if `Balance` > 0) | ✅ | ✅ | the information in `Balance` lies in zero vs positive; replaces the amount |
| `NumOfProducts` grouped 1 / 2 / 3+ | ✅ (One-Hot) | ✅ (One-Hot) | U-shape, rare categories 3 and 4 (see Encoding) |
| `Age` centred + its square | ✅ | ❌ (`Age` as is) | inverted U: churn peaks at 50–60 and drops after 60; centering reduces the correlation between `Age` and `Age²` |
| `has_balance` × `NumOfProducts` | ✅ | ❌ | the effect of `has_balance` reverses with the number of products; a tree captures interactions natively |

**Age for the logistic regression: both `Age` centred (`Age_c`) and its square (`Age_c_sq`).**
- `Age_c` gives the slope (does churn rise or fall with age?), `Age_c_sq` the curvature (inverted U).
- `Age_c_sq` alone would force a curve symmetric around the mean age (≈ 38): the peak would be at 38, while the data show it at 50–60. With both terms, the model places the peak where the data put it.
- Centring makes `Age_c` and `Age_c_sq` almost uncorrelated (raw `Age` and `Age²` are correlated at ≈ 0.99), which gives more stable coefficients. Predictions are the same either way.
- In the pipeline: `StandardScaler(with_std=False)` centres `Age` (mean computed on `train`), then a `FunctionTransformer` returns both `Age_c` and `Age_c_sq`. Both columns are kept; they are scaled later, at the normalisation step.

**Not done for now:** `is_germany` (the One-Hot of `Geography` keeps the full information), age groups, polynomial features (to keep the number of variables small; to be tested later if needed).

**Hypothesis:** the decision tree may outperform the logistic regression because of the non-linearities found in the EDA. To be checked with cross-validated F1, both models being tuned with the same care.

### 4. Feature selection

**Dropped before selection:** `ID`, `CustomerId`, `Surname` (identifiers, no business meaning) and `Balance` (replaced by `has_balance`).

**Candidates:** `Geography`, `Gender`, `NumOfProducts` (1 / 2 / 3+), `IsActiveMember`, `has_balance`, `Age`, `Tenure`, `CreditScore`, `EstimatedSalary`, `HasCrCard`. For the logistic regression, the engineered features are also candidates: `Age` centered, `Age²` and `has_balance` × `NumOfProducts`.

**Method:** `SelectKBest` with `mutual_info_classif` (captures non-linear links such as the U-shapes), **inside the pipeline**, with `k` tuned by `GridSearchCV` on `train` (it uses the target, so fitting it outside cross-validation would leak).

**Limits:** `SelectKBest` scores each column alone, so it mostly confirms the EDA. The list of selected columns is checked afterwards (e.g. an interaction kept without its original variables is hard to interpret).

**`Gender`:** excluded from the final model for ethical reasons (unequal treatment of customers by gender), whatever `SelectKBest` returns. It is kept here only to check whether it would be selected: being predictive does not make it acceptable.

### 5. Normalisation

| Variable | Logistic regression | Reason |
|---|---|---|
| `Age_c` and `Age_c_sq` | `StandardScaler` | built at the feature engineering step (centred age and its square); scaled here like the other continuous variables |
| `CreditScore` | `StandardScaler` | roughly normal, with some extreme low values (`MinMaxScaler` would be sensitive to them) |
| `EstimatedSalary` | `MinMaxScaler` | bounded (0 to 200,000) and almost uniform |
| `Tenure` | `MinMaxScaler` | already an ordinal code (0 to 10), bounded; scaled so that its range matches the 0/1 columns |
| binary and One-Hot columns | none (0/1) | usual practice |

Scaling matters for the logistic regression: its default regularization penalizes large coefficients, so a variable's scale changes the result. **Decision tree: no scaling** (it splits on thresholds, the scale does not change anything).

## Pipelines

The pipelines follow the course order: cleaning, encoding, feature engineering, feature selection, normalisation. Everything learned from the data (encoders, centring, selection, scalers, model) is inside a `Pipeline`, so it is fitted on the training folds only during cross-validation and on `train` only at the end.

| Step | What is done | Logistic regression | Decision tree | Tool |
|---|---|---|---|---|
| **1. Cleaning** | Drop `ID`, `CustomerId`, `Surname` (no business meaning); no missing values; outliers kept | ✅ | ✅ | `ColumnTransformer` (`"drop"`) |
| **2. Encoding** | a) Rare categories: 3 and 4 products → "3+" (`NumOfProducts` → `products`) | ✅ | ✅ | `ColumnTransformer` + function |
| | b) One-Hot of `Geography`, `products` | without the reference category (`drop="first"`) | full (all categories) | `OneHotEncoder` |
| | c) `Gender` as a single 0/1 column (`Gender_Male`) | ✅ | ✅ | `OneHotEncoder(drop="if_binary")` |
| **3. Feature engineering** | a) `has_balance` = 1 if `Balance` > 0 (replaces `Balance`) | ✅ | ✅ | `ColumnTransformer` + function |
| | b) Interactions `has_balance` × `products_2`, `has_balance` × `products_3+` | ✅ | ❌ (a tree finds interactions itself) | `ColumnTransformer` + function |
| | c) Centre `Age`, then square it → `Age_c`, `Age_c_sq` | ✅ | ❌ (`Age` as is) | `ColumnTransformer`: `StandardScaler(with_std=False)` + function |
| **4. Feature selection** | Keep the k columns most linked to churn | ✅ | ✅ | `SelectKBest(mutual_info)` |
| **5. Normalisation** | `Age_c`, `Age_c_sq`, `CreditScore` | `StandardScaler` | ❌ (no scaling for a tree) | `StandardScaler` |
| | `EstimatedSalary`, `Tenure` | `MinMaxScaler` | ❌ | `MinMaxScaler` |
| | 0/1 columns | unchanged | unchanged | — |
| **6. Model** | Predict | `LogisticRegression` | `DecisionTreeClassifier` | — |

Normalisation is applied to the columns **kept** by the selection.

**Centring is not normalisation:** at step 3, age is only centred (mean of `train` subtracted) so that `Age_c` and `Age_c_sq` are not almost identical (raw `Age` and `Age²` are correlated at ≈ 0.99). Scaling happens at step 5.

In [116]:
X_train = train.drop(columns="Exited")
y_train = train["Exited"]
X_test = test.drop(columns="Exited")
y_test = test["Exited"]

In [117]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, FunctionTransformer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier


# ---------- Step 1. Cleaning ----------
# Identifiers have no business meaning (CustomerId and Surname are not real identifiers, see 01_eda)
cleaning = ColumnTransformer(
    [("drop_identifiers", "drop", ["ID", "CustomerId", "Surname"])],
    remainder="passthrough",          # all other columns are kept
    verbose_feature_names_out=False,
)


# ---------- Step 2. Encoding ----------
def group_products(x):
    """Rare categories: 3 and 4 products (2% of customers, same churn) are grouped into "3+"."""
    return x.clip(upper=3).replace({1: "1", 2: "2", 3: "3+"})


def products_name(transformer, input_features):
    return ["products"]


rare_categories = ColumnTransformer(
    [("group_rare_products", FunctionTransformer(group_products, feature_names_out=products_name), ["NumOfProducts"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
)


def make_encoding(drop_first):
    """One-Hot of Geography and products (drop_first=True removes the reference category);
    Gender as a single 0/1 column (drop="if_binary")."""
    onehot = OneHotEncoder(drop="first" if drop_first else None, sparse_output=False)
    return Pipeline([
        ("rare_categories", rare_categories),
        ("onehot", ColumnTransformer([
            ("onehot", onehot, ["Geography", "products"]),
            ("binary", OneHotEncoder(drop="if_binary", sparse_output=False), ["Gender"]),
        ], remainder="passthrough", verbose_feature_names_out=False)),
    ])


# ---------- Step 3. Feature engineering ----------
def balance_flag(x):
    """has_balance = 1 if Balance > 0 (replaces Balance)."""
    return (x > 0).astype(int)


def has_balance_name(transformer, input_features):
    return ["has_balance"]


def make_has_balance():
    return ColumnTransformer(
        [("has_balance", FunctionTransformer(balance_flag, feature_names_out=has_balance_name), ["Balance"])],
        remainder="passthrough",
        verbose_feature_names_out=False,
    )


def add_interactions(x):
    """has_balance x products (1 product is the reference category); input columns are kept."""
    x = x.copy()
    x["balance_x_products_2"] = x["has_balance"] * x["products_2"]
    x["balance_x_products_3+"] = x["has_balance"] * x["products_3+"]
    return x


def interaction_names(transformer, input_features):
    return list(input_features) + ["balance_x_products_2", "balance_x_products_3+"]


interactions = ColumnTransformer(
    [("balance_x_products", FunctionTransformer(add_interactions, feature_names_out=interaction_names),
      ["has_balance", "products_2", "products_3+"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
)


def with_square(x):
    """Return the centred column and its square, side by side."""
    return np.hstack([x, x ** 2])


def square_names(transformer, input_features):
    return [f"{input_features[0]}_c", f"{input_features[0]}_c_sq"]


age_square = ColumnTransformer(
    [("age", Pipeline([
        ("center", StandardScaler(with_std=False)),   # Age - mean of train (centring only)
        ("square", FunctionTransformer(with_square, feature_names_out=square_names)),
    ]), ["Age"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
)

feature_engineering_logit = Pipeline([
    ("has_balance", make_has_balance()),
    ("interactions", interactions),
    ("age", age_square),
])

feature_engineering_tree = Pipeline([
    ("has_balance", make_has_balance()),
])


# ---------- Step 4. Feature selection ----------
def mutual_info(X, y):
    """Mutual information with each column, declaring discrete columns (at most 20 distinct values).
    By default, mutual_info_classif treats every column as continuous, which is biased for 0/1 columns."""
    X = np.asarray(X)
    discrete = np.array([len(np.unique(X[:, j])) <= 20 for j in range(X.shape[1])])
    return mutual_info_classif(X, y, discrete_features=discrete, random_state=42)


# ---------- Step 5. Normalisation (on the columns kept by the selection) ----------
# make_column_selector takes only the listed columns that are still present: the selection
# (step 4) may have removed some of them, and a fixed list would then raise an error.
normalisation_logit = ColumnTransformer([
    ("standard", StandardScaler(), make_column_selector(pattern="^(?:Age_c|Age_c_sq|CreditScore)$")),
    ("minmax", MinMaxScaler(), make_column_selector(pattern="^(?:EstimatedSalary|Tenure)$")),
], remainder="passthrough", verbose_feature_names_out=False)   # 0/1 columns unchanged


# ---------- Pipelines ----------
logit_pipe = Pipeline([
    ("cleaning", cleaning),
    ("encoding", make_encoding(drop_first=True)),
    ("feature_engineering", feature_engineering_logit),
    ("selection", SelectKBest(mutual_info, k="all")),
    ("normalisation", normalisation_logit),
    ("model", LogisticRegression(max_iter=1000)),
])

tree_pipe = Pipeline([
    ("cleaning", cleaning),
    ("encoding", make_encoding(drop_first=False)),
    ("feature_engineering", feature_engineering_tree),
    ("selection", SelectKBest(mutual_info, k="all")),
    ("model", DecisionTreeClassifier(random_state=42)),   # no normalisation for a tree
])

**Pipeline diagrams** (click on a step to see its content).

In [118]:
logit_pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('cleaning', ...), ('encoding', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('drop_identifiers', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"verbose_feature_names_out verbose_feature_names_o

In [119]:
tree_pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('cleaning', ...), ('encoding', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('drop_identifiers', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"verbose_feature_names_out verbose_feature_names_o

**Check:** columns produced by each pipeline before feature selection (fitted on `train`).

In [120]:
# Columns produced before feature selection (steps 1 to 3), fitted on train
X_prepared_logit = logit_pipe[:3].fit_transform(X_train, y_train)
X_prepared_tree = tree_pipe[:3].fit_transform(X_train, y_train)
print("logistic regression:", X_prepared_logit.shape, "| decision tree:", X_prepared_tree.shape)
print(list(X_prepared_logit.columns))
print(list(X_prepared_tree.columns))

logistic regression: (114863, 15) | decision tree: (114863, 14)
['Age_c', 'Age_c_sq', 'has_balance', 'products_2', 'products_3+', 'balance_x_products_2', 'balance_x_products_3+', 'Geography_Germany', 'Geography_Spain', 'Gender_Male', 'CreditScore', 'Tenure', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
['has_balance', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'products_1', 'products_2', 'products_3+', 'Gender_Male', 'CreditScore', 'Age', 'Tenure', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']


**SelectKBest ranking** (mutual information between each column and `Exited`, computed on `train`).

This ranking is descriptive: the number of columns kept (`k`) will be tuned by cross-validation at the modeling step, not chosen by looking at this table.

In [121]:
rankings = {}
for name, pipe in [("logistic regression", logit_pipe), ("decision tree", tree_pipe)]:
    pipe[:4].fit(X_train, y_train)   # cleaning, encoding, feature engineering, selection
    selection = pipe.named_steps["selection"]
    rankings[name] = pd.Series(selection.scores_, index=selection.feature_names_in_).sort_values(ascending=False)

pd.concat(rankings, axis=1).round(4)

,logistic regression,decision tree
Age_c_sq,0.0827,NaN
Age_c,0.0824,NaN
products_2,0.0770,0.0770
IsActiveMember,0.0228,0.0228
products_3+,0.0213,0.0213
Geography_Germany,0.0206,0.0206
balance_x_products_3+,0.0128,NaN
Gender_Male,0.0107,0.0107
has_balance,0.0090,0.0090
EstimatedSalary,0.0070,0.0073


**Ranking consistent with the EDA.** The top columns are the variables most linked to churn in `01_eda`:
- `Age_c_sq` and `Age_c` first (≈ 0.08): age, the largest gap (0.83 standard deviation) with an inverted U;
- `products_2` (0.077) and `products_3+`: `NumOfProducts`, the strongest categorical link (V = 0.42), U-shaped;
- `IsActiveMember` (0.023) and `Geography_Germany` (0.021): both V = 0.21.
- At the bottom: `HasCrCard`, `Tenure`, `CreditScore`, negligible in the EDA. `Geography_Spain` is also low: Spain behaves like France (the reference), so the country effect lies entirely in Germany.

- `Gender_Male` ranks in the middle (8th of 15, ≈ 0.011): with k ≥ 8, `SelectKBest` would keep it. It is predictive, but it is **excluded from the final model for ethical reasons** (a targeting model should not treat customers differently by gender, see `00_framing`).
Two different methods (statistical tests in the EDA, mutual information here) lead to the same conclusions.